# Episodic CEBC / BANKING77


In [ ]:
import hashlib
import json
import platform
import re
import sys
from collections import Counter, defaultdict
from importlib import metadata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import display
from sklearn.model_selection import train_test_split

DATASET = 'banking77'
N_WAY = 5
K_SUPPORT = 2
N_QUERY = 2
ALLOW_SELF_PAIRS = False
MAX_EPISODES = 3000
MAX_LENGTH = 64
LEARNING_RATE = 5e-5
BATCH_SIZE = 16
VALID_SUPPORT_PER_CLASS = 2
EXPECTED_CLASSES = 77
EXPECTED_SIZES = (8996, 1000, 3080)
BACKBONE = "bert-base-uncased"
SEED = 42
SHOT_BUDGET = None
FINAL_TEST = False
RUN_CEBC_API_SMOKE = False

cwd = Path.cwd()
PROJECT = cwd.parent if cwd.name == "notebooks" else cwd
CLINC_SHA256 = "36923c3705a59e08fe9c3883d8bc2dd966ef93e22cb78ac41171782a698d56e0"
NORMALIZATION = "strip/collapse whitespace and Unicode casefold; punctuation retained"

packages = ("torch", "transformers", "datasets", "accelerate", "numpy", "pandas",
            "scikit-learn", "matplotlib", "ipykernel")
versions = {name: metadata.version(name) for name in packages}
print("Interpreter:", sys.executable)
print("Python:", platform.python_version(), "| versions:", versions)
print("CUDA build:", torch.version.cuda, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


c:\APP\Anaconda\envs\NLP_CW1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Interpreter: c:\APP\Anaconda\envs\NLP_CW1\python.exe
Python: 3.10.22 | versions: {'torch': '2.11.0+cu128', 'transformers': '4.44.2', 'datasets': '2.21.0', 'accelerate': '0.34.2', 'numpy': '2.2.6', 'pandas': '2.3.3', 'scikit-learn': '1.7.2', 'matplotlib': '3.10.9', 'ipykernel': '7.3.0'}
CUDA build: 12.8 | CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## Reuse the audited data protocol

The preprocessing and label helpers below come from the existing project and are inlined. The official Test rows are used by the protocol for leakage checks. They are reserved from training and model selection.

In [2]:
def normalize_text(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Text must be a non-empty string")
    return re.sub(r'\s+', ' ', text).strip().casefold()

def text_hash(text):
    return hashlib.sha256(normalize_text(text).encode("utf-8")).hexdigest()

def audit_overlap(left, right, left_name, right_name):
    a = left.assign(_key=left["text"].map(normalize_text))
    b = right.assign(_key=right["text"].map(normalize_text))
    # Keep the original audit column order for the saved 01 evidence.
    pairs = a.drop(columns="text_sha256", errors="ignore").merge(
        b.drop(columns="text_sha256", errors="ignore"),
        on="_key", suffixes=("_left", "_right")
    )
    report = {
        "left": left_name, "right": right_name,
        "matching_pairs": len(pairs), "unique_texts": int(pairs["_key"].nunique()),
        "left_rows": int(pairs["source_id_left"].nunique()),
        "right_rows": int(pairs["source_id_right"].nunique()),
        "conflicting_pairs": int((pairs["label_left"] != pairs["label_right"]).sum()),
    }
    return report, pairs

def remove_train_overlap(train, protected, name):
    report, pairs = audit_overlap(train, protected, name, "protected")
    drop_ids = set(pairs["source_id_left"])
    cleaned = train.loc[~train["source_id"].isin(drop_ids)].copy()
    assert len(train) - len(cleaned) == report["left_rows"]
    print(name, "removed training rows:", len(drop_ids),
          "conflicting pairs:", report["conflicting_pairs"])
    return cleaned, report, pairs

def verify_disjoint(a, b, name_a, name_b):
    overlap = set(a["text_sha256"]) & set(b["text_sha256"])
    assert not overlap, f"Normalized-text overlap: {name_a} / {name_b}"
    assert set(a["source_id"]).isdisjoint(b["source_id"])
    return {"left": name_a, "right": name_b, "unique_texts": len(overlap)}

def make_source_ids(df, dataset_name, split_name):
    prefix = "banking77" if dataset_name == "banking77" else "clinc"
    result = df.copy()
    result["source_id"] = [f"{prefix}_{split_name}_{i}" for i in range(len(result))]
    result["text_sha256"] = result["text"].map(text_hash)
    return result

def read_original_clinc():
    """Read optional local original data, or download in memory; never export JSON."""
    path = PROJECT / "data" / "clinc150" / "data_full.json"
    if path.exists():
        content = path.read_bytes()
    else:
        import urllib.request
        url = "https://raw.githubusercontent.com/clinc/oos-eval/master/data/data_full.json"
        content = urllib.request.urlopen(url, timeout=90).read()
    assert hashlib.sha256(content).hexdigest() == CLINC_SHA256, "Original CLINC SHA256 mismatch"
    return json.loads(content)

def sample_fewshot(pool, shot_budget, seed, sampling_mode):
    """Sample a few-shot set; nested budgets share per-class random prefixes."""
    if sampling_mode not in {"independent", "nested"}:
        raise ValueError("sampling_mode must be 'independent' or 'nested'")
    if shot_budget is None:
        return pool.copy()
    if type(shot_budget) is not int or shot_budget <= 0:
        raise ValueError("shot_budget must be None or a positive integer")
    if sampling_mode == "nested" and shot_budget > 16:
        raise ValueError("nested sampling supports shot budgets up to 16")
    assert pool.groupby("label").size().min() >= shot_budget, "Insufficient examples per class"
    if sampling_mode == "independent":
        # Preserve legacy pandas sample behavior, IDs and row order exactly.
        return pool.groupby("label", group_keys=False).sample(
            n=shot_budget, random_state=seed).copy()

    rng = np.random.RandomState(seed)
    chosen_ids = []
    for label in sorted(pool["label"].unique().tolist()):
        class_ids = pool.loc[pool["label"] == label, "source_id"].to_numpy()
        chosen_ids.extend(rng.permutation(class_ids)[:shot_budget].tolist())
    selected_ids = set(chosen_ids)
    return pool.loc[pool["source_id"].isin(selected_ids)].copy()

def build_protocol(dataset_name, shot_budget=8, seed=42, validation_fraction=0.10,
                   sampling_mode="independent"):
    if dataset_name not in {"banking77", "clinc150"}:
        raise ValueError("Choose banking77 or clinc150")
    if shot_budget is not None and (type(shot_budget) is not int or shot_budget <= 0):
        raise ValueError("shot_budget must be None or a positive integer")
    if sampling_mode not in {"independent", "nested"}:
        raise ValueError("sampling_mode must be 'independent' or 'nested'")
    if sampling_mode == "nested" and shot_budget is not None and shot_budget > 16:
        raise ValueError("nested sampling supports shot budgets up to 16")
    if type(seed) is not int or not 0 <= seed < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32)")
    if not 0 < validation_fraction < 1:
        raise ValueError("validation_fraction must be between zero and one")
    audits = {}
    pairs = {}
    oos_audit = []
    original_sha256 = None
    if dataset_name == "banking77":
        raw = load_dataset("PolyAI/banking77")
        id_to_name = dict(enumerate(raw["train"].features["label"].names))
        expected_classes = 77
        train = make_source_ids(raw["train"].to_pandas(), dataset_name, "train")
        test = make_source_ids(raw["test"].to_pandas(), dataset_name, "test")
        cleaned, audits["train_vs_test"], pairs["train_vs_test"] = remove_train_overlap(train, test, "banking_train")
        pool, validation = train_test_split(
            cleaned, test_size=validation_fraction, random_state=seed,
            stratify=cleaned["label"]
        )
        pool, audits["pool_vs_validation"], pairs["pool_vs_validation"] = remove_train_overlap(pool, validation, "banking_pool")
        original_splits = {"train": train, "test": test}
        expected_labels = set(id_to_name)
    else:
        original = read_original_clinc()
        original_sha256 = CLINC_SHA256
        converted = load_dataset("DeepPavlov/clinc150", "default")
        id_to_names = defaultdict(set)
        original_splits = {}
        all_original_texts = {text for rows in original.values() for text, label in rows}
        for split, raw_name in [("train", "train"), ("validation", "val"), ("test", "test")]:
            raw_rows = original[raw_name]
            text_to_names = defaultdict(set)
            for text, name in raw_rows:
                text_to_names[text].add(name)
            assert all(len(names) == 1 for names in text_to_names.values()), "Ambiguous original labels"
            hf_frame = converted[split].to_pandas()
            known = hf_frame.loc[hf_frame["label"].notna()]
            assert Counter(known["utterance"]) == Counter(text for text, name in raw_rows)
            for text, number in zip(known["utterance"], known["label"]):
                id_to_names[int(number)].update(text_to_names[text])
            missing = Counter(hf_frame.loc[hf_frame["label"].isna(), "utterance"])
            oos = Counter(text for text, name in original["oos_" + raw_name])
            unknown = missing - oos
            assert not (oos - missing), "Original OOS rows missing from converted split"
            oos_audit.append({
                "split": split, "null_labels": sum(missing.values()),
                "original_oos_matches": sum((missing & oos).values()),
                "unconfirmed_extra_rows": sum(unknown.values()),
                "extra_rows_in_any_original_split": sum(n for text, n in unknown.items() if text in all_original_texts),
            })
            frame = pd.DataFrame(raw_rows, columns=["text", "label"])
            # Preserve the established IDs: i indexes original JSON in-domain rows.
            original_splits[split] = make_source_ids(frame, dataset_name, split)
        assert len(id_to_names) == 150 and all(len(names) == 1 for names in id_to_names.values())
        id_to_name = {number: next(iter(names)) for number, names in id_to_names.items()}
        assert set(id_to_name) == set(range(150)) and len(set(id_to_name.values())) == 150
        expected_classes = 150
        expected_labels = set(id_to_name.values())
        train = original_splits["train"]
        validation = original_splits["validation"]
        test = original_splits["test"]
        pool, audits["train_vs_validation"], pairs["train_vs_validation"] = remove_train_overlap(train, validation, "clinc_train")
        pool, audits["train_vs_test_after_validation"], pairs["train_vs_test_after_validation"] = remove_train_overlap(pool, test, "clinc_train")

    checks = []
    for a, b, an, bn in [
        (pool, validation, "pool", "validation"),
        (pool, test, "pool", "test"),
        (validation, test, "validation", "test"),
    ]:
        checks.append(verify_disjoint(a, b, an, bn))
    for frame, frame_name in [(pool, "pool"), (validation, "validation"), (test, "test")]:
        assert set(frame["label"]) == expected_labels, f"Unexpected labels in {frame_name}"
        assert frame["source_id"].is_unique, f"Duplicate source_id in {frame_name}"
        assert frame["text_sha256"].equals(frame["text"].map(text_hash)), f"Hash mismatch in {frame_name}"
    source = pd.concat(original_splits.values()).set_index("source_id")
    assert source.index.is_unique, "Duplicate source_id in original source rows"
    for frame in [pool, validation, test]:
        restored = source.loc[frame["source_id"], ["text", "label", "text_sha256"]].reset_index(drop=True)
        pd.testing.assert_frame_equal(restored, frame[["text", "label", "text_sha256"]].reset_index(drop=True))
    # Official protected content and row order must remain complete.
    pd.testing.assert_frame_equal(test, original_splits["test"])
    if dataset_name == "clinc150":
        pd.testing.assert_frame_equal(validation, original_splits["validation"])
    fewshot = sample_fewshot(pool, shot_budget, seed, sampling_mode)
    if shot_budget is not None:
        counts = fewshot.groupby("label").size()
        assert len(counts) == expected_classes and counts.eq(shot_budget).all(), (
            "Few-shot budget/class coverage mismatch")
    assert fewshot["source_id"].is_unique
    pool_by_id = pool.set_index("source_id")
    restored_fewshot = pool_by_id.loc[
        fewshot["source_id"], ["text", "label", "text_sha256"]
    ].reset_index(drop=True)
    pd.testing.assert_frame_equal(
        restored_fewshot,
        fewshot[["text", "label", "text_sha256"]].reset_index(drop=True),
    )
    meta = {
        "protocol_version": "2.0",
        "dataset": dataset_name, "shot_budget": shot_budget, "seed": seed,
        "sampling_mode": sampling_mode,
        "nested_max_shot": 16 if sampling_mode == "nested" else None,
        "validation_fraction": validation_fraction if dataset_name == "banking77" else None,
        "normalization": NORMALIZATION, "hash_algorithm": "SHA256 of normalized UTF-8 text",
        "split_sizes": {"pool": len(pool), "validation": len(validation), "test": len(test),
                        "fewshot": len(fewshot) if shot_budget is not None else None},
        "fewshot_note": "When shot_budget is None, fewshot is a pool copy for compatibility.",
        "classes": expected_classes, "overlap_audits": audits,
        "disjoint_checks": checks, "oos_audit": oos_audit,
        "clinc_original_sha256": original_sha256,
        "id_to_name": id_to_name,
        "source_id_order": "HF original split rows" if dataset_name == "banking77" else "Original CLINC JSON in-domain split rows (legacy protocol)",
    }
    return {"pool": pool, "validation": validation, "test": test, "fewshot": fewshot,
            "protocol_meta": meta, "audit_pairs": pairs}

def validate_training_protocol(protocol):
    """Verify selected training IDs and content using the current seed and budget."""
    meta = protocol["protocol_meta"]
    offer = protocol["pool"]
    selected = protocol["fewshot"]
    budget = meta["shot_budget"]
    mode = meta["sampling_mode"]
    seed = meta["seed"]
    if mode not in {"independent", "nested"}:
        raise ValueError("sampling_mode must be 'independent' or 'nested'")
    print(
        "Protocol metadata:",
        {"protocol_version": meta.get("protocol_version"),
         "sampling_mode": mode,
         "nested_max_shot": meta.get("nested_max_shot"),
         "split_sizes": meta.get("split_sizes")},
    )
    for name in ("pool", "validation", "test", "fewshot"):
        frame = protocol[name]
        assert frame["source_id"].is_unique, f"Duplicate source_id in {name}"
        assert frame["text_sha256"].equals(frame["text"].map(text_hash)), f"Hash mismatch in {name}"

    for left_name, right_name in [
        ("pool", "validation"), ("pool", "test"), ("validation", "test")
    ]:
        verify_disjoint(protocol[left_name], protocol[right_name], left_name, right_name)

    offer_by_id = offer.set_index("source_id")
    expected_hashes = selected["source_id"].map(offer_by_id["text_sha256"])
    assert expected_hashes.notna().all(), "Few-shot ID outside pool"
    assert selected["text_sha256"].reset_index(drop=True).equals(
        expected_hashes.reset_index(drop=True)), "Few-shot hash differs from pool"
    restored_selected = offer_by_id.loc[
        selected["source_id"], ["text", "label"]
    ].reset_index(drop=True)
    pd.testing.assert_frame_equal(
        restored_selected, selected[["text", "label"]].reset_index(drop=True)
    )

    if budget is None:
        pd.testing.assert_frame_equal(selected, offer)
    else:
        expected = sample_fewshot(offer, budget, seed, mode)
        assert set(selected["source_id"]) == set(expected["source_id"]), (
            "Few-shot IDs differ from current seed/budget/mode")
    return True

def encode_labels(protocol, train, evaluation):
    """Encode labels in a stable contiguous space derived from the full pool."""
    original_labels = sorted(protocol["pool"]["label"].unique().tolist())
    original_to_class = {label: index for index, label in enumerate(original_labels)}
    train_labels = train["label"].map(original_to_class)
    actual = evaluation["label"].map(original_to_class)
    if train_labels.isna().any() or actual.isna().any():
        raise ValueError("Train/evaluation labels must be represented in the pool")
    train_labels = train_labels.to_numpy(dtype=np.int64)
    actual = actual.to_numpy(dtype=np.int64)
    n_classes = len(original_labels)
    if (not original_labels or len(np.unique(train_labels)) != n_classes or
            np.any((train_labels < 0) | (train_labels >= n_classes)) or
            np.any((actual < 0) | (actual >= n_classes))):
        raise ValueError("Encoded train/evaluation labels are incomplete or out of range")
    return original_labels, original_to_class, train_labels, actual

In [3]:
protocol = build_protocol(dataset_name=DATASET, shot_budget=SHOT_BUDGET, seed=SEED)
validate_training_protocol(protocol)
pool, validation = protocol["pool"], protocol["validation"]
original_labels, original_to_class, train_y, val_y = encode_labels(protocol, pool, validation)
id_to_intent = {i: protocol["protocol_meta"]["id_to_name"][label]
                for i, label in enumerate(original_labels)}
split_names = ("pool", "validation", "test")
assert tuple(len(protocol[split]) for split in split_names) == EXPECTED_SIZES
assert set(train_y) == set(val_y) == set(id_to_intent) == set(range(EXPECTED_CLASSES))
assert len(set(id_to_intent.values())) == EXPECTED_CLASSES
display(pd.DataFrame([
    {"split": split, "rows": len(protocol[split]), "classes": protocol[split]["label"].nunique()}
    for split in split_names
]))
print("Source ID order:", protocol["protocol_meta"]["source_id_order"])
display(pd.DataFrame(protocol["protocol_meta"]["overlap_audits"]).T)
if DATASET == "clinc150":
    assert set(id_to_intent.values()) == set(protocol["protocol_meta"]["id_to_name"].values())
    print("CLINC150 original JSON SHA256:", protocol["protocol_meta"]["clinc_original_sha256"])
    display(pd.DataFrame(protocol["protocol_meta"]["oos_audit"]))
print("Data preparation only. No episodes, model training, or evaluation have run.")


banking_train removed training rows: 7 conflicting pairs: 0
banking_pool removed training rows: 0 conflicting pairs: 0
Protocol metadata: {'protocol_version': '2.0', 'sampling_mode': 'independent', 'nested_max_shot': None, 'split_sizes': {'pool': 8996, 'validation': 1000, 'test': 3080, 'fewshot': None}}


,split,rows,classes
0,pool,8996,77
1,validation,1000,77
2,test,3080,77


Source ID order: HF original split rows


,left,right,matching_pairs,unique_texts,left_rows,right_rows,conflicting_pairs
train_vs_test,banking_train,protected,7,7,7,7,0
pool_vs_validation,banking_pool,protected,0,0,0,0,0


Data preparation only. No episodes, model training, or evaluation have run.


## Reproducibility and guards for episode and Support inputs

The protocol above already audits split overlaps, labels and row counts. This cell sets seeds and adds two independent checks you can call when your sampler and fixed Support selection are ready. `assert_pool_roles` respects the self-pair policy you choose. `assert_fixed_support` verifies every provided class label against its Pool Source ID, and checks that every intent has equal Support coverage.

In [4]:
import random

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if FINAL_TEST:
    raise ValueError("The development notebook only prepares the protocol and API checks. Set FINAL_TEST=False.")

# Shared protocol already verifies split separation and sizes. Bind the encoded
# training labels to the same Source IDs for subsequent CEBC safety checks.
pool_class_by_id = dict(zip(pool["source_id"], map(int, train_y), strict=True))
assert len(pool_class_by_id) == len(pool)

def assert_pool_roles(support_ids, query_ids, *, allow_self_pairs):
    """Check concrete training episode roles; sampling and overlap policy are supplied by the caller."""
    support_ids, query_ids = tuple(support_ids), tuple(query_ids)
    if not support_ids or not query_ids:
        raise ValueError("An episode needs both support and query items.")
    for role, ids in (("Support", support_ids), ("Query", query_ids)):
        outside = set(ids).difference(pool_class_by_id)
        if outside:
            raise ValueError(f"Training {role} contains IDs outside Pool: {sorted(outside)[:3]}")
    if not allow_self_pairs and not set(support_ids).isdisjoint(query_ids):
        raise ValueError("This episode uses the same Source ID as both Support and Query.")

def assert_fixed_support(support_ids, support_class_ids):
    """Validate a supplied closed-set Support bank without reading Validation labels."""
    ids = tuple(support_ids)
    labels = tuple(support_class_ids)
    if len(ids) != len(labels) or len(ids) != len(set(ids)):
        raise ValueError("Support Source IDs must be unique and aligned with their labels.")
    for source_id, reported_label in zip(ids, labels, strict=True):
        if source_id not in pool_class_by_id or pool_class_by_id[source_id] != reported_label:
            raise ValueError(f"Support item is outside Pool or has the wrong label: {source_id}")
    counts = Counter(labels)
    if set(counts) != set(id_to_intent) or len(set(counts.values())) != 1:
        raise ValueError("Support bank must contain the same positive number of samples for every intent.")
    return next(iter(counts.values()))

print("Protocol validation complete. Source-ID role checks are ready for the sampler and Support bank.")


Protocol validation complete. Source-ID role checks are ready for the sampler and Support bank.


## Optional synthetic check of the model API

Set `RUN_CEBC_API_SMOKE=True` in setup to run this diagnostic. It uses the cached `bert-base-uncased` tokenizer and a randomly initialized one-layer BERT. It checks Query-major pair ordering, binary targets, BCE parity and gradients on four invented pairs. It makes no optimizer update and produces no real-data predictions or scores.

If the tokenizer is unavailable, cache it once with `AutoTokenizer.from_pretrained("bert-base-uncased")` in `NLP_CW1`. The formal experiment will also need pretrained BERT model weights. The diagnostic's model is local to this function.

In [5]:
def check_cebc_api():
    from torch import nn
    from transformers import AutoTokenizer, BertConfig, BertModel

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    query = ["Please reset my password.", "I need a taxi."]
    support = ["A cab is needed.", "Change the password."]
    query_labels = torch.tensor([0, 1], device=device)
    support_labels = torch.tensor([1, 0], device=device)
    tokenizer = AutoTokenizer.from_pretrained(BACKBONE, local_files_only=True)
    pairs = tokenizer([q for q in query for _ in support], support * len(query),
                      padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
    assert pairs.input_ids.shape[0] == 4
    assert pairs.token_type_ids.eq(1).any(dim=1).all().item(), "No second sequence in the pair encoding."
    targets = query_labels[:, None].eq(support_labels[None, :]).float()
    expected = torch.tensor([[0., 1.], [1., 0.]], device=device)
    assert torch.equal(targets, expected), "Pair order and binary labels disagree."

    devices = [device.index] if device.type == "cuda" else []
    with torch.random.fork_rng(devices=devices):
        torch.manual_seed(SEED)
        encoder = BertModel(BertConfig(vocab_size=len(tokenizer), hidden_size=32,
                                      num_hidden_layers=1, num_attention_heads=4,
                                      intermediate_size=64)).to(device)
        head = nn.Sequential(nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.5), nn.Linear(16, 1)).to(device)
        pooled = encoder(**pairs).pooler_output
        assert pooled.shape == (4, 32)
        logits = head(pooled).reshape(len(query), len(support))
        loss = nn.functional.binary_cross_entropy_with_logits(logits, targets)
        original_form = nn.functional.binary_cross_entropy(logits.sigmoid(), targets)
        assert torch.isfinite(loss).item() and torch.allclose(loss, original_form, atol=1e-6, rtol=1e-6)
        loss.backward()
        for name, module in (("encoder", encoder), ("similarity head", head)):
            gradients = [p.grad for p in module.parameters() if p.grad is not None]
            assert gradients and all(torch.isfinite(g).all().item() for g in gradients), name
            assert any(torch.count_nonzero(g).item() for g in gradients), name

        encoder.eval()
        head.eval()
        with torch.inference_mode():
            scores = head(encoder(**pairs).pooler_output).reshape(len(query), len(support))
            pred = support_labels[scores.argmax(dim=1)]
            assert pred.shape == (len(query),)
    print(f"Synthetic CEBC API check passed on {device}: {len(query)} queries, {len(support)} supports, "
          "2 positive and 2 negative pairs, and finite gradients. No optimizer step.")

if RUN_CEBC_API_SMOKE:
    check_cebc_api()
else:
    print("Synthetic CEBC API check disabled. Enable RUN_CEBC_API_SMOKE to run it.")


Synthetic CEBC API check disabled. Enable RUN_CEBC_API_SMOKE to run it.


## Student implementation checklist

- **Pair encoder.** With `bert-base-uncased`, the tokenizer takes two lists: `tokenizer(query_texts, support_texts, padding=True, truncation=True, return_tensors="pt")`. A joint forward pass yields BERT's `pooler_output`. The CEBC head maps `hidden_size → hidden_size // 2 → 1` using ReLU and dropout 0.5. Match sigmoid + BCE, or use `BCEWithLogitsLoss` on raw logits. Each target is 1 for equal intent IDs and 0 otherwise. [Encoder](https://github.com/UKPLab/eacl2023-few-shot-intent-classification/blob/9489edfe30b3a612abd5a0bf49aafb7fc33e8f34/encoder.py) · [Head](https://github.com/UKPLab/eacl2023-few-shot-intent-classification/blob/9489edfe30b3a612abd5a0bf49aafb7fc33e8f34/classifier.py)
- **Choose and describe the training variant.** The authors' combined-set script permits each example to pair with itself. The separate support/query script uses separate groups and can omit self-pairs with disjoint Source IDs. Set `N_WAY`, `K_SUPPORT`, `N_QUERY` per class, the episode budget and sampling rule. Record unique Pool rows visited, coverage, repeated sampling, pair counts, optimizer steps, time and peak allocated GPU memory. [Combined-set script](https://github.com/UKPLab/eacl2023-few-shot-intent-classification/blob/9489edfe30b3a612abd5a0bf49aafb7fc33e8f34/episodic_learning.py) · [Separate support/query script](https://github.com/UKPLab/eacl2023-few-shot-intent-classification/blob/9489edfe30b3a612abd5a0bf49aafb7fc33e8f34/episodic_learning_wsq.py)
- **Specify a fixed Pool-only support bank.** Pick and document the seed, selection rule and number K of supports per class before evaluating. Check 77 candidate classes and record support Source IDs. Batch pair inference and take the label of the maximum-scoring individual support for each query. This is equivalent to classwise max followed by argmax when ties use the same support order. Validation contains 1,000 queries, which is 77,000 × K pairs. The scorer takes query text, without query labels. Use Validation labels afterward for metrics and error tables. [CEBC prediction](https://github.com/UKPLab/eacl2023-few-shot-intent-classification/blob/9489edfe30b3a612abd5a0bf49aafb7fc33e8f34/model.py)
- **Retain results inside the notebook.** Display the actual settings and metrics, class support and F1, confusion pairs and Source-ID error examples. Keep Test disabled during model selection. Record train and inference time and reset peak GPU allocation for each. Keep experiment results in memory and in cell outputs.

This project adaptation keeps a fixed closed-set label space and Pool-only supports. Mesgar et al. used an episodic few-shot protocol. Attribute the paper and upstream repository when describing the method. The upstream `LICENSE` is Apache-2.0 at commit `9489edfe30b3a612abd5a0bf49aafb7fc33e8f34` while its README text says MIT. If you reuse source code, resolve that discrepancy before redistribution and retain the relevant license, NOTICE, copyright and modification notices. Upstream NOTICE credits UKP Lab, Technische Universität Darmstadt. This starter copies project protocol helpers and provides API notes. [Paper](https://aclanthology.org/2023.eacl-main.135/) · [LICENSE](https://github.com/UKPLab/eacl2023-few-shot-intent-classification/blob/9489edfe30b3a612abd5a0bf49aafb7fc33e8f34/LICENSE) · [NOTICE](https://github.com/UKPLab/eacl2023-few-shot-intent-classification/blob/9489edfe30b3a612abd5a0bf49aafb7fc33e8f34/NOTICE.txt) · [README](https://github.com/UKPLab/eacl2023-few-shot-intent-classification/blob/9489edfe30b3a612abd5a0bf49aafb7fc33e8f34/README.md)

**Budget check:** If separate roles use `N_WAY` classes, `K_SUPPORT` supports and `N_QUERY` queries per class, each balanced episode produces `N_WAY**2 * K_SUPPORT * N_QUERY` pairs. `N_WAY * K_SUPPORT * N_QUERY` are positive. The combined-set variant has a different pair count and includes the diagonal. Verify the actual count and record it alongside Pool coverage. When Validation scores are batched, keep all classes eligible for every query and verify exactly one prediction per Source ID.